# Model Training Runner

This notebook drives `src/model_training.py` end-to-end.

**Steps covered:**
1. Resolve project root and configure paths
2. Verify data files exist
3. Run the training script
4. Inspect saved metrics

## Step 1 — Resolve project root and configure paths

`git rev-parse --show-toplevel` gives the absolute repo root regardless of where
Jupyter was launched, so all paths below are stable.

In [1]:
import subprocess
import sys
from pathlib import Path

from src.paths import MODEL_DATA_DIR, MODELS_DIR

PROJECT_ROOT = subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True
).strip()

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

TRAIN = MODEL_DATA_DIR / "df_train_final.parquet"
VALID = MODEL_DATA_DIR / "df_valid_final.parquet"
TEST = MODEL_DATA_DIR / "df_test_final.parquet"
OUT_DIR = str(MODELS_DIR)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"OUT_DIR      : {OUT_DIR}")

PROJECT_ROOT : D:/AI/Real projects/Academic_Advisor
OUT_DIR      : D:\AI\Real projects\Academic_Advisor\models


## Step 2 — Verify data files exist

In [2]:
for path in [TRAIN, VALID, TEST]:
    p = Path(path)
    status = "OK" if p.exists() else "MISSING"
    print(f"[{status}] {path}")

[OK] D:\AI\Real projects\Academic_Advisor\data\model_data\df_train_final.parquet
[OK] D:\AI\Real projects\Academic_Advisor\data\model_data\df_valid_final.parquet
[OK] D:\AI\Real projects\Academic_Advisor\data\model_data\df_test_final.parquet


## Step 3 — Run the training script

Calls `src.model_training.main()` directly so output streams into the notebook.

> The script will:
> - **STEP 8.5** sanity-check `final_mark` (existence, no nulls, values in [0, 100])
> - Train the LightGBM grade regression model (MAE loss)
> - Train the LightGBM pass/fail classifier (binary AUC)
> - Run stratified AUC breakdown (skips segments with only one class)
> - Save `grade_model.lgbm`, `pass_model.lgbm`, and `metrics.json` to `OUT_DIR`

In [3]:
import json

# Patch sys.argv so argparse inside main() picks up our absolute paths.
sys.argv = [
    "model_training",
    "--train", TRAIN,
    "--valid", VALID,
    "--test",  TEST,
    "--out",   OUT_DIR,
]

from src.model_training import main
main()

TypeError: 'WindowsPath' object is not subscriptable

## Step 4 — Inspect saved metrics

In [ ]:
metrics_path = Path(OUT_DIR) / "metrics.json"
with open(metrics_path) as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2))

## Alternative — run as CLI command

From the project root in a terminal:

```bash
python -m src.model_training \
    --train $(python -c "from src.paths import MODEL_DATA_DIR; print(MODEL_DATA_DIR / 'df_train_final.parquet')") \
    --valid $(python -c "from src.paths import MODEL_DATA_DIR; print(MODEL_DATA_DIR / 'df_valid_final.parquet')") \
    --test  $(python -c "from src.paths import MODEL_DATA_DIR; print(MODEL_DATA_DIR / 'df_test_final.parquet')") \
    --out   $(python -c "from src.paths import MODELS_DIR; print(MODELS_DIR)")
```

Or use environment variables to set `ACADEMIC_ADVISOR_DATA_DIR` before running.